# SDSS Local Manager: Pipeline Demonstration

This notebook demonstrates how to use the `sdss_manager` library to automatically sync a subset of the Sloan Digital Sky Survey (SDSS) data, index its astronomical metadata into a local SQLite database, and query the catalog cleanly.

In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd

from sdss_manager.core import SDSSDatabase

# Jupyter-specific Pandas configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

ModuleNotFoundError: No module named 'sdss_manager'

## 1. Initialize and Update the Database
We instantiate the `SDSSDatabase` manager, pointing it to a local data directory and a metadata database file. Calling `update_database()` will trigger the differential Rsync sync and automatically index any newly arrived FITS files.

In [ ]:
# Define paths
DATA_DIR = "../data/sdss_subset"
DB_PATH = "../sdss_metadata.db"

# Initialize the library wrapper
db = SDSSDatabase(local_base_dir=DATA_DIR, db_path=DB_PATH)

# Run the pipeline (Syncs missing files + extracts FITS headers into SQL)
db.update_database()

## 2. Inspecting the Parsed Catalog
Instead of scanning gigabytes of FITS files every time, this library queries the indexed SQLite database instantly. Load this into a Pandas DataFrame for rich analysis.

In [ ]:
# Fetch the indexed catalog as a dataframe
df = db.get_catalog()

print(f"Total observations indexed: {len(df)}")

# Style the output: format floats, handle missing values, and highlight key columns
styled_df = (
    df.head(10)
    .style.format(
        {"ra": "{:.4f}°", "dec": "{:.4f}°", "exposure_time": "{:.1f}s"}
    )
    .highlight_max(axis=0, subset=["exposure_time"], color="#e6f2ff")
    .set_caption("SDSS Metadata Catalog (First 10 Observations)")
)

# Display the styled table natively in Jupyter
styled_df

## 3. Data Verification & Visualizations
Plot the spatial distribution (Right Ascension vs. Declination) of our downloaded targets, mapping the color intensity to their exposure times. This confirms coordinate parsing logic is entirely accurate.

In [ ]:
if not df.empty:
    plt.figure(figsize=(10, 5), dpi=100)

    # Create a scatter plot of the sky coordinates
    scatter = plt.scatter(
        df["ra"],
        df["dec"],
        c=df["exposure_time"],
        cmap="plasma",
        alpha=0.7,
        edgecolors="none",
        s=40,
    )

    # Initialize chart
    plt.title(
        "Spatial Distribution of Parsed SDSS Observations",
        fontsize=14,
        pad=15,
    )
    plt.xlabel("Right Ascension (RA)", fontsize=11)
    plt.ylabel("Declination (Dec)", fontsize=11)
    plt.grid(True, linestyle="--", alpha=0.5)

    # Colorbar to represent exposure depth
    cbar = plt.colorbar(scatter)
    cbar.set_label("Exposure Time (seconds)", rotation=270, labelpad=15)

    plt.tight_layout()
    plt.show()
else:
    print("No data available to plot. Ensure files are properly indexed.")